# Configurazione ambiente e caricamento modello pre-addestrato per sentiment analysis con funzione di inferenza semplice

Questa cella configura l'ambiente di lavoro installando le librerie essenziali,
carica un modello pre-addestrato per l'analisi del sentiment specifico per testi
da Twitter, e definisce una funzione di inferenza semplice che prende in input
una frase testuale e restituisce la distribuzione di probabilità tra sentiment
negativo, neutro e positivo. Infine, viene testata la funzione con un esempio
di frase per verificarne il funzionamento.


In [ ]:
# Installiamo le librerie necessarie
!pip install transformers --quiet
!pip install torch --quiet
!pip install evaluate
!pip install --upgrade transformers

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# Nome del modello su HuggingFace
model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"

# Carichiamo tokenizer e modello
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Funzione di inferenza semplice
def predict_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    scores = outputs.logits.softmax(dim=1).squeeze()
    labels = ['negative', 'neutral', 'positive']
    # Creiamo un dizionario per le probabilità predette
    probs = {label: float(score) for label, score in zip(labels, scores)}
    return probs

# Test con un esempio di frase
test_text = "I am really happy with the new product!"
print(predict_sentiment(test_text))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 95.0 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.2
    Uninstalling transformers-4.57.2:
      Successfully uninstalled transformers-4.57.2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


{'negative': 0.0027311549056321383, 'neutral': 0.0068116700276732445, 'positive': 0.9904572367668152}


# Caricamento e visualizzazione iniziale del dataset tweet_eval per sentiment analysis

Caricamento e prima esplorazione del dataset tweet_eval, utile per
comprendere la struttura dei dati e preparare il terreno per il fine-tuning.

In [ ]:
!pip install datasets --quiet

from datasets import load_dataset

# Carichiamo il dataset tweet_eval per task sentiment
dataset = load_dataset("tweet_eval", "sentiment")

# Visualizziamo le prime righe del train set
print(dataset['train'][0:5])

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentiment/train-00000-of-00001.parquet:   0%|          | 0.00/3.78M [00:00<?, ?B/s]

sentiment/test-00000-of-00001.parquet:   0%|          | 0.00/901k [00:00<?, ?B/s]

sentiment/validation-00000-of-00001.parq(…):   0%|          | 0.00/167k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/45615 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/12284 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

{'text': ['"QT @user In the original draft of the 7th book, Remus Lupin survived the Battle of Hogwarts. #HappyBirthdayRemusLupin"', '"Ben Smith / Smith (concussion) remains out of the lineup Thursday, Curtis #NHL #SJ"', 'Sorry bout the stream last night I crashed out but will be on tonight for sure. Then back to Minecraft in pc tomorrow night.', "Chase Headley's RBI double in the 8th inning off David Price snapped a Yankees streak of 33 consecutive scoreless innings against Blue Jays", '@user Alciato: Bee will invest 150 million in January, another 200 in the Summer and plans to bring Messi by 2017"'], 'label': [2, 1, 1, 1, 2]}


# Tokenizzazione del dataset tweet_eval e preparazione dei DataLoader per il training

In questa cella applichiamo la tokenizzazione ai testi del dataset utilizzando il tokenizer associato al modello pre-addestrato. Configuriamo la formattazione dei dati in tensori PyTorch e creiamo DataLoader per il training e la validazione, preparando così i dati per la fase di fine-tuning del modello.

In [ ]:
from transformers import AutoTokenizer
from torch.utils.data import DataLoader

model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Funzione di tokenizzazione dei testi
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

# Applichiamo la tokenizzazione sul dataset
tokenized_datasets = dataset.map(preprocess_function, batched=True)

# Settiamo il formato PyTorch
tokenized_datasets.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# Creiamo DataLoader per train e validation
train_loader = DataLoader(tokenized_datasets["train"], batch_size=16, shuffle=True)
val_loader = DataLoader(tokenized_datasets["validation"], batch_size=16)

print("Tokenizzazione e preparazione DataLoader completati.")

Map:   0%|          | 0/45615 [00:00<?, ? examples/s]

Map:   0%|          | 0/12284 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenizzazione e preparazione DataLoader completati.


In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
import torch

def infer_batch(batch, model, device):
    model.to(device)
    model.eval()
    with torch.no_grad():
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=1)
    return preds.cpu()


In [ ]:
import torch

all_preds = []
all_labels = []

for batch in val_loader:
    preds = infer_batch(batch, model, device)  # Funzione definita precedentemente
    labels = batch["label"].to(device)         # Spostiamo le label sul device
    all_preds.append(preds)
    all_labels.append(labels.cpu())             # Torniamo label su CPU per confronto

# Concatenazione di tutte le predizioni e label
all_preds_tensor = torch.cat(all_preds)
all_labels_tensor = torch.cat(all_labels)

# Calcolo accuracy
accuracy = (all_preds_tensor == all_labels_tensor).sum().item() / len(all_labels_tensor)
print(f"Accuracy sul validation set: {accuracy:.4f}")

Accuracy sul validation set: 0.7715


# Definizione della pipeline di training con Hugging Face Trainer

In questo passaggio configuriamo i parametri di training tramite TrainingArguments, definiamo una funzione per il calcolo dell’accuratezza come metrica di valutazione, associamo i dataset tokenizzati al Trainer e avviamo il processo di fine-tuning del modello. Al termine effettuiamo una valutazione finale sul set di validazione per misurare le prestazioni del modello ottimizzato.



In [ ]:
from transformers import Trainer, TrainingArguments
import evaluate
import numpy as np

# 1. Impostiamo il device per spostare i dati/modello se necessario

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Definizione degli argomenti per il training
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",       # Valutazione a fine epoca
    save_strategy="epoch",       # Salvataggio a fine epoca (Match obbligatorio!)
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=2,          # Tiene solo i 2 checkpoint migliori per risparmiare spazio
    load_best_model_at_end=True, # Ricarica il modello migliore alla fine
    metric_for_best_model="accuracy",
    report_to="none"
)

# 3. Carichiamo la metrica accuracy (usando evaluate)
metric = evaluate.load("accuracy")

# 4. Funzione di valutazione da passare a Trainer
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# 5. Prepariamo i dataset train e validation (sotto forma di Dataset Hugging Face)
#train_dataset = tokenized_datasets["train"]
#eval_dataset = tokenized_datasets["validation"]

# 6. Creiamo l'istanza Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"], # USA QUI
    eval_dataset=tokenized_datasets["validation"], # E QUI
    compute_metrics=compute_metrics,
)

# 7. Avviamo il training
trainer.train()

# 8. Valutazione finale sul validation set
results = trainer.evaluate()
print(f"Risultati di valutazione sul validation set: {results}")

Epoch,Training Loss,Validation Loss,Accuracy
1,0.572100,0.576642,0.752500
2,0.416900,0.683672,0.755000
3,0.300700,0.993670,0.760000


Risultati di valutazione sul validation set: {'eval_loss': 0.9936696887016296, 'eval_accuracy': 0.76, 'eval_runtime': 12.8234, 'eval_samples_per_second': 155.965, 'eval_steps_per_second': 19.496, 'epoch': 3.0}


In [ ]:
import torch
print("Device CUDA disponibile:", torch.cuda.is_available())
print("Nome GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Nessuna GPU")

Device CUDA disponibile: True
Nome GPU: Tesla T4


#CONDIVISIONE LINK REPOSITORY GITHUB

Per la realizzazione del progetto di MLOps sono partito dalla stesura del codice presente nella parte superiore di questo Colab.

Successivamente ho lavorato su Github. Lascio il link alla repository relativa al progetto:

https://github.com/RG17-o/MachineInnovators